# Scenario 2: Seller quality

Business question: which sellers are reliably bad for the marketplace, and
what do they cost it?

"Reliably" is the hard part. A seller with 3 reviews averaging 1.7 might be
fine and unlucky. We rank on a statistic that penalises small samples.

Run from the project root:  python analysis/02_sellers.py
Reads:  order_items_clean.csv, orders_clean.csv, reviews_dedup.csv, sellers_clean.csv
Writes: outputs/exports/sellers_*.csv  and  outputs/figures/sellers_*.png

In [1]:
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(14)

polars.config.Config

In [3]:
MIN_REVIEWED_ORDERS = 20      # a seller needs this many reviewed orders to be ranked at all
WORST_N = 20                  # how many sellers to name on the slide

## 1. Load

In [4]:
items = pl.read_csv("data/cleaned/order_items_clean.csv")
orders = pl.read_csv("data/cleaned/orders_clean.csv", try_parse_dates=True)
reviews = pl.read_csv("data/cleaned/reviews_dedup.csv")
sellers = pl.read_csv("data/cleaned/sellers_clean.csv",
                      schema_overrides={"seller_zip_code_prefix": pl.String})

In [5]:
items.shape, orders.shape, reviews.shape, sellers.shape

((112650, 7), (99441, 14), (98673, 7), (3095, 4))

## 2. One row per (seller, order)
`order_items` has one row per item. An order with 3 items from the same seller
would give that seller 3 copies of the same review. So first collapse to one
row per seller per order, summing the money.

In [6]:
seller_orders = (
    items.group_by("seller_id", "order_id")
    .agg(
        pl.len().alias("n_items"),
        pl.col("price").sum().alias("revenue"),
        pl.col("freight_value").sum().alias("freight"),
    )
)
seller_orders.shape                       # fewer rows than items: multi-item orders collapsed

(100010, 5)

## 3. Attach order status, delivery result, and the review
Both joins are on `order_id` against tables that have one row per order, so
the row count cannot change.

In [7]:
seller_orders = seller_orders.join(
    orders.select("order_id", "order_status", "valid_for_delay_calc", "delivery_status"),
    on="order_id", how="left",
)
seller_orders = seller_orders.join(reviews.select("order_id", "review_score"), on="order_id", how="left")
seller_orders.shape

(100010, 9)

In [8]:
seller_orders = seller_orders.with_columns(
    (pl.col("review_score") <= 2).alias("is_bad_review"),      # null stays null when there is no review
    (pl.col("order_status") == "canceled").alias("is_canceled"),
    (pl.col("delivery_status") == "late").alias("is_late"),
)
seller_orders.head(3)

seller_id,order_id,n_items,revenue,freight,order_status,valid_for_delay_calc,delivery_status,review_score,is_bad_review,is_canceled,is_late
str,str,u32,f64,f64,str,bool,str,i64,bool,bool,bool
"""da8622b14eb17a…","""186d50e82bb60f…",1,44.9,8.47,"""delivered""",true,"""on_time""",3,false,false,false
"""9de4643a8dbde6…","""38953e88e1f6d0…",1,167.99,16.94,"""delivered""",true,"""on_time""",5,false,false,false
"""710e3548e02bc1…","""3dc1f7f4445977…",1,79.9,29.49,"""delivered""",true,"""on_time""",4,false,false,false


## 4. Aggregate to one row per seller
`is_late` is only meaningful for delivered orders with valid timestamps, so
the late rate is computed on that subset with a filter inside the aggregation.

In [9]:
per_seller = (
    seller_orders.group_by("seller_id")
    .agg(
        pl.len().alias("orders"),
        pl.col("review_score").is_not_null().sum().alias("reviewed_orders"),
        pl.col("review_score").mean().round(3).alias("mean_score"),
        pl.col("is_bad_review").sum().alias("bad_reviews"),
        pl.col("is_canceled").sum().alias("canceled_orders"),
        pl.col("is_late").filter(pl.col("valid_for_delay_calc") & (pl.col("order_status") == "delivered")).mean().round(4).alias("late_rate"),
        pl.col("revenue").sum().round(2).alias("revenue"),
    )
    .with_columns(
        (pl.col("bad_reviews") / pl.col("reviewed_orders")).round(4).alias("bad_share"),
        (pl.col("canceled_orders") / pl.col("orders")).round(4).alias("cancel_rate"),
    )
)
per_seller.shape

(3095, 10)

In [10]:
per_seller.sort("orders", descending=True).head(5)

seller_id,orders,reviewed_orders,mean_score,bad_reviews,canceled_orders,late_rate,revenue,bad_share,cancel_rate
str,u32,u32,f64,u32,u32,f64,f64,f64,f64
"""6560211a19b479…",1854,1838,3.934,319,7,0.0532,123304.83,0.1736,0.0038
"""4a3ca9315b744c…",1806,1785,3.828,342,2,0.0979,200472.92,0.1916,0.0011
"""cc419e0650a3c5…",1706,1698,4.069,256,9,0.0529,104288.42,0.1508,0.0053
"""1f50f920176fa8…",1404,1399,4.133,198,1,0.0891,106939.21,0.1415,0.0007
"""da8622b14eb17a…",1314,1308,4.176,155,0,0.0677,160236.57,0.1185,0.0


## 5. The trap this ranking avoids
Sort by raw bad_share and the top of the list is sellers with 1 or 2 reviews.

In [11]:
per_seller.filter(pl.col("bad_share") == 1.0).sort("reviewed_orders").select("seller_id", "reviewed_orders", "bad_reviews", "bad_share").head(8)

seller_id,reviewed_orders,bad_reviews,bad_share
str,u32,u32,f64
"""c7b7db6c8f3c64…",1,1,1.0
"""c85d7b477a709c…",1,1,1.0
"""1a6245add4353f…",1,1,1.0
"""f9eda05b67bef4…",1,1,1.0
"""154bdf805377af…",1,1,1.0
"""c360e4787614ed…",1,1,1.0
"""0336182e1b3e92…",1,1,1.0
"""ca5832c6960267…",1,1,1.0


In [12]:
per_seller.filter(pl.col("bad_share") == 1.0)["reviewed_orders"].value_counts().sort("reviewed_orders").head(6)

reviewed_orders,count
u32,u32
1,138
2,35
3,5
4,1


## 6. Wilson lower bound
For a proportion (here: share of reviewed orders that got 1 or 2 stars), the
Wilson score interval gives a range the true share probably sits in.
Its lower bound is what we rank on: "we are 95% sure at least this share of
their orders got a bad review". Few reviews means a wide interval, so a
2-review seller with a 100% bad share gets a lower bound near 0.2, while a
60-review seller at 40% bad gets a lower bound near 0.29 and ranks above them.

Formula, with p = bad_share, n = reviewed_orders, z = 1.96:
  (p + z^2/2n - z * sqrt(p(1-p)/n + z^2/4n^2)) / (1 + z^2/n)

In [13]:
z = 1.96
p = pl.col("bad_share")
n = pl.col("reviewed_orders")
wilson_low = (p + z**2 / (2 * n) - z * ((p * (1 - p) / n) + z**2 / (4 * n**2)).sqrt()) / (1 + z**2 / n)

In [14]:
per_seller = per_seller.with_columns(wilson_low.round(4).alias("bad_share_lower_bound"))
per_seller.select("seller_id", "reviewed_orders", "bad_share", "bad_share_lower_bound").sort("bad_share_lower_bound", descending=True).head(5)

seller_id,reviewed_orders,bad_share,bad_share_lower_bound
str,u32,f64,f64
"""bcd2d7510d58e2…",0,NaN,NaN
"""80ceebb4ee9b31…",0,NaN,NaN
"""400f221ab83037…",0,NaN,NaN
"""20f0aeea30bc3b…",0,NaN,NaN
"""3820c6537b3853…",0,NaN,NaN


## 7. Rank only sellers with enough reviews
The Wilson bound already penalises small n, but a hard floor keeps the list
defensible: every named seller has at least 20 reviewed orders.

In [15]:
ranked = (
    per_seller.filter(pl.col("reviewed_orders") >= MIN_REVIEWED_ORDERS)
    .sort("bad_share_lower_bound", descending=True)
    .with_row_index("rank", offset=1)
)
ranked.shape                              # how many sellers qualify

(811, 12)

In [16]:
ranked = ranked.join(sellers.select("seller_id", "seller_state", "seller_city"), on="seller_id", how="left")
worst = ranked.head(WORST_N)
worst.select("rank", "seller_id", "seller_state", "reviewed_orders", "bad_share", "bad_share_lower_bound", "late_rate", "cancel_rate", "revenue")

rank,seller_id,seller_state,reviewed_orders,bad_share,bad_share_lower_bound,late_rate,cancel_rate,revenue
u32,str,str,u32,f64,f64,f64,f64,f64
1,"""1ca7077d890b90…","""SP""",114,0.614,0.5223,0.1667,0.0,13341.57
2,"""ffff564a4f9085…","""SP""",20,0.65,0.4329,0.0,0.2,1426.3
3,"""2eb70248d66e0e…","""SP""",198,0.5,0.431,0.1123,0.0,42628.61
4,"""54965bbe3e4f07…","""PR""",74,0.4459,0.3381,0.3099,0.0,10961.3
5,"""a49928bcdf77c5…","""SP""",98,0.4082,0.3161,0.2188,0.0,8816.7
6,"""972d0f9cf61b49…","""SC""",79,0.4177,0.3152,0.1111,0.0,8089.29
7,"""b19f3ca2ea4759…","""MG""",24,0.5,0.3143,0.3158,0.0417,6202.91
8,"""bbad7e518d7af8…","""SP""",68,0.4118,0.3026,0.1642,0.0,4462.22
9,"""d71d863e5ef30d…","""SP""",26,0.4615,0.2875,0.1304,0.0,11900.9


## 8. What do these sellers cost?
Compare the worst 20 to everyone else who qualified.

In [17]:
rest = ranked.filter(pl.col("rank") > WORST_N)
comparison = pl.DataFrame({
    "group": [f"worst_{WORST_N}", "other_ranked_sellers"],
    "sellers": [worst.height, rest.height],
    "orders": [worst["orders"].sum(), rest["orders"].sum()],
    "revenue": [round(worst["revenue"].sum(), 2), round(rest["revenue"].sum(), 2)],
    "bad_share": [round(worst["bad_reviews"].sum() / worst["reviewed_orders"].sum(), 4),
                  round(rest["bad_reviews"].sum() / rest["reviewed_orders"].sum(), 4)],
    "late_rate": [round(float(worst["late_rate"].mean()), 4), round(float(rest["late_rate"].mean()), 4)],
    "cancel_rate": [round(worst["canceled_orders"].sum() / worst["orders"].sum(), 4),
                    round(rest["canceled_orders"].sum() / rest["orders"].sum(), 4)],
})
comparison

group,sellers,orders,revenue,bad_share,late_rate,cancel_rate
str,i64,i64,f64,f64,f64,f64
"""worst_20""",20,1608,264879.1,0.4091,0.1616,0.0106
"""other_ranked_s…",791,85992,1.0883e7,0.1396,0.0634,0.003


In [18]:
share_of_all_bad_reviews = worst["bad_reviews"].sum() / ranked["bad_reviews"].sum()
share_of_all_orders = worst["orders"].sum() / ranked["orders"].sum()
round(share_of_all_bad_reviews * 100, 1), round(share_of_all_orders * 100, 1)

(5.2, 1.8)

## 9. Are bad sellers bad because they are late?
If bad_share tracks late_rate, the fix is logistics. If a seller has a high
bad share and a normal late rate, the problem is the product or the service.

In [19]:
ranked.select(pl.corr("late_rate", "bad_share").alias("corr_late_vs_bad"))

corr_late_vs_bad
f64
0.528946


In [20]:
worst.select("rank", "seller_id", "bad_share", "late_rate").with_columns(
    pl.when(pl.col("late_rate") > ranked["late_rate"].median() * 2).then(pl.lit("lateness"))
    .otherwise(pl.lit("product or service"))
    .alias("likely_cause")
)

rank,seller_id,bad_share,late_rate,likely_cause
u32,str,f64,f64,str
1,"""1ca7077d890b90…",0.614,0.1667,"""lateness"""
2,"""ffff564a4f9085…",0.65,0.0,"""product or ser…"
3,"""2eb70248d66e0e…",0.5,0.1123,"""lateness"""
4,"""54965bbe3e4f07…",0.4459,0.3099,"""lateness"""
5,"""a49928bcdf77c5…",0.4082,0.2188,"""lateness"""
6,"""972d0f9cf61b49…",0.4177,0.1111,"""lateness"""
7,"""b19f3ca2ea4759…",0.5,0.3158,"""lateness"""
8,"""bbad7e518d7af8…",0.4118,0.1642,"""lateness"""
9,"""d71d863e5ef30d…",0.4615,0.1304,"""lateness"""


## 10. Export

In [21]:
ranked.write_csv("outputs/exports/sellers_ranked.csv")
worst.write_csv("outputs/exports/sellers_worst.csv")
comparison.write_csv("outputs/exports/sellers_comparison.csv")

## 11. Figures

In [22]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh([f"#{r} ({s[:8]})" for r, s in zip(worst["rank"].to_list(), worst["seller_id"].to_list())],
        (worst["bad_share_lower_bound"] * 100).to_list(), color="#C44E52")
ax.invert_yaxis()
ax.set_xlabel("Bad-review share, 95% lower bound (%)")
ax.set_title(f"{WORST_N} sellers with the most certain bad-review share (min {MIN_REVIEWED_ORDERS} reviewed orders)")
for i, (lb, n) in enumerate(zip(worst["bad_share_lower_bound"].to_list(), worst["reviewed_orders"].to_list())):
    ax.text(lb * 100 + 0.5, i, f"n={n}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("outputs/figures/sellers_worst_by_wilson.png", dpi=150)
plt.close()

In [23]:
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter((ranked["late_rate"] * 100).to_list(), (ranked["bad_share"] * 100).to_list(),
           s=12, alpha=0.4, color="#4C72B0", label="ranked sellers")
ax.scatter((worst["late_rate"] * 100).to_list(), (worst["bad_share"] * 100).to_list(),
           s=40, color="#C44E52", label=f"worst {WORST_N}")
ax.set_xlabel("Late delivery rate (%)")
ax.set_ylabel("Bad-review share (%)")
ax.set_title("Sellers: lateness vs bad reviews")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/figures/sellers_late_vs_bad.png", dpi=150)
plt.close()

In [24]:
print("Scenario 2 done.")
print(f"  sellers with any order: {per_seller.height:,}   ranked (>= {MIN_REVIEWED_ORDERS} reviewed): {ranked.height:,}")
print(f"  worst {WORST_N}: {share_of_all_orders*100:.1f}% of ranked orders, {share_of_all_bad_reviews*100:.1f}% of ranked bad reviews")
print(f"  worst {WORST_N} bad share {comparison['bad_share'][0]*100:.1f}% vs others {comparison['bad_share'][1]*100:.1f}%")

Scenario 2 done.
  sellers with any order: 3,095   ranked (>= 20 reviewed): 811
  worst 20: 1.8% of ranked orders, 5.2% of ranked bad reviews
  worst 20 bad share 40.9% vs others 14.0%
